In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path(r"C:\Users\ghkdr\OneDrive\바탕 화면\이스트캠프\데이콘\credit_default")
DATA_DIR = BASE / "data"
FE_DIR = DATA_DIR / "fe"

FE_DIR.mkdir(exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print("FE_DIR:", FE_DIR)

DATA_DIR: C:\Users\ghkdr\OneDrive\바탕 화면\이스트캠프\데이콘\credit_default\data
FE_DIR: C:\Users\ghkdr\OneDrive\바탕 화면\이스트캠프\데이콘\credit_default\data\fe


In [9]:
train = pd.read_csv(DATA_DIR / "open" / "train.csv")
test  = pd.read_csv(DATA_DIR / "open" / "test.csv")

print(train.shape, test.shape)
train.head()

(26457, 20) (10000, 19)


,index,gender,car,reality,child_num,income_total,income_type,edu_type,family_type,house_type,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,work_phone,phone,email,occyp_type,family_size,begin_month,credit
0,0,F,N,N,0,202500.0,Commercial associate,Higher education,Married,Municipal apartment,-13899,-4709,1,0,0,0,NaN,2.0,-6.0,1.0
1,1,F,N,Y,1,247500.0,Commercial associate,Secondary / secondary special,Civil marriage,House / apartment,-11380,-1540,1,0,0,1,Laborers,3.0,-5.0,1.0
2,2,M,Y,Y,0,450000.0,Working,Higher education,Married,House / apartment,-19087,-4434,1,0,1,0,Managers,2.0,-22.0,2.0
3,3,F,N,Y,0,202500.0,Commercial associate,Secondary / secondary special,Married,House / apartment,-15088,-2092,1,0,1,0,Sales staff,2.0,-37.0,0.0
4,4,F,Y,Y,0,157500.0,State servant,Higher education,Married,House / apartment,-15037,-2105,1,0,0,0,Managers,2.0,-26.0,2.0


In [10]:
# 1) 이상치 제거
train = train[train["family_size"] <= 7].reset_index(drop=True)

# 2) 불필요 컬럼 제거
drop_cols = ["index", "FLAG_MOBIL"]
train = train.drop(columns=drop_cols)
test  = test.drop(columns=drop_cols)

# 3) DAYS_EMPLOYED 특수값 처리
train["DAYS_EMPLOYED"] = train["DAYS_EMPLOYED"].apply(lambda x: 0 if x > 0 else x)
test["DAYS_EMPLOYED"]  = test["DAYS_EMPLOYED"].apply(lambda x: 0 if x > 0 else x)

# 4) 날짜 절대값
for col in ["DAYS_BIRTH", "DAYS_EMPLOYED", "begin_month"]:
    train[col] = train[col].abs()
    test[col]  = test[col].abs()

print(train.shape, test.shape)

(26451, 18) (10000, 17)


In [11]:
for df in [train, test]:
    df["before_EMPLOYED"] = df["DAYS_BIRTH"] - df["DAYS_EMPLOYED"]
    df["Age"] = df["DAYS_BIRTH"] // 365
    df["EMPLOYED"] = df["DAYS_EMPLOYED"] // 365
    
    df["income_per_family"] = df["income_total"] / df["family_size"]
    df["ability"] = df["income_total"] / (df["DAYS_BIRTH"] + df["DAYS_EMPLOYED"])

In [12]:
train["income_total"] = np.log1p(train["income_total"])
test["income_total"]  = np.log1p(test["income_total"])

In [13]:
y = train["credit"]
X_train = train.drop(columns=["credit"])
X_test = test.copy()

print(X_train.shape, X_test.shape)

(26451, 22) (10000, 22)


In [14]:
train_fe = pd.concat([X_train, y], axis=1)
test_fe = X_test.copy()

train_fe.to_csv(FE_DIR / "train_fe_v3_clean.csv", index=False)
test_fe.to_csv(FE_DIR / "test_fe_v3_clean.csv", index=False)

print("Saved.")

Saved.


In [15]:
print(train_fe.shape)
print(test_fe.shape)

(26451, 23)
(10000, 22)
